# A Practical Primer on Dates, Times, and Calendars in Python (with TM1py notes)

## 1. The mental model

Before any code, internalize one distinction: **an instant** (a point on the universal timeline) versus **civil time** (what a wall clock and calendar say). The same instant is `2024-03-10 07:00 UTC`, `2024-03-10 03:00` in New York (EDT), and `2024-03-10 02:00` in Chicago (CST, before DST). When data crosses systems — spreadsheets, databases, TM1 cubes, REST APIs — most bugs come from confusing these two views.

A second distinction: **calendar periods** (year, quarter, month, fiscal week) versus **timestamps** (exact moments). TM1 time dimensions are calendar periods — elements like `2024`, `Q1-2024`, `Jan-2024`, `WK01-2024`. Pandas, by contrast, mostly works with timestamps and ranges over them.

Most data-processing pain is translating between these two views.

## 2. The standard library

The core module is `datetime`. It defines four types you'll use constantly:

- `datetime.date` — a calendar date (year, month, day), no time, no zone.
- `datetime.time` — a time of day (hour, minute, second, microsecond), no date.
- `datetime.datetime` — date + time, optionally with a timezone.
- `datetime.timedelta` — a duration; the result of subtracting two datetimes or adding/subtracting from one.

In [ ]:
from datetime import date, datetime, time, timedelta, timezone

today = date.today()
now = datetime.now()                   # local naive
now_utc = datetime.now(timezone.utc)   # aware, in UTC
delta = timedelta(days=7, hours=3)
next_week = now + delta

There is also `datetime.timezone` for fixed offsets and `zoneinfo.ZoneInfo` (Python 3.9+) for IANA zones. Use `ZoneInfo` whenever you need real-world zones with DST behavior.

The older `time` module is mostly for low-level Unix-epoch interop (`time.time()` returns seconds since 1970-01-01 UTC as a float) and sleeping. Use it sparingly — `datetime` covers almost everything cleanly.

## 3. Naive vs. aware datetimes

This is the single most important rule:

- A **naive** datetime has no timezone. It's just numbers — Python cannot tell whether `2024-03-10 02:30` is in Tokyo or São Paulo, or even whether it's a valid wall-clock time (DST gaps make some times nonexistent).
- An **aware** datetime carries a `tzinfo` object and represents an unambiguous instant.

Mixing them is forbidden in arithmetic — Python raises `TypeError` when you subtract one from the other — but you can still compare, format, and store them inadvertently, which is where bugs hide.

**Recommendation for data pipelines:** store and pass instants as aware UTC datetimes; convert to local zones only at display boundaries.

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# Get an aware UTC instant
now_utc = datetime.now(timezone.utc)

# Convert to a civil zone for display
now_ny = now_utc.astimezone(ZoneInfo("America/New_York"))

# Attach a zone to a naive datetime that you *know* is in that zone
naive = datetime(2024, 3, 10, 9, 30)
aware = naive.replace(tzinfo=ZoneInfo("America/New_York"))  # naive really is NY local

`replace(tzinfo=...)` does not convert; it asserts. Use `astimezone(...)` to convert.

## 4. Parsing and formatting

`datetime.strptime(s, fmt)` parses a string with an explicit format. `datetime.strftime(fmt)` does the reverse. The format codes follow C conventions: `%Y` four-digit year, `%m` zero-padded month, `%d` day, `%H` 24-hour, `%M` minute, `%S` second, `%z` UTC offset, `%Z` zone name, `%b` short month name (`Jan`), `%B` full month name.

In [ ]:
datetime.strptime("2024-03-10 09:30", "%Y-%m-%d %H:%M")
datetime.strptime("10/03/2024", "%d/%m/%Y").date()

For ISO 8601, Python 3.11+ has a much-improved `datetime.fromisoformat`:

In [ ]:
datetime.fromisoformat("2024-03-10T09:30:00+00:00")

For messy real-world strings, `dateutil.parser.parse` is permissive and handy, but slow and ambiguous (it'll guess between `dd/mm` and `mm/dd`). Be explicit when you can.

For Unix epoch values, both stdlib and pandas know what to do:

In [ ]:
datetime.fromtimestamp(1709974800, tz=timezone.utc)   # stdlib
pd.to_datetime(1709974800, unit="s", utc=True)        # pandas

## 5. Date arithmetic

`timedelta` handles fixed durations (days, seconds, microseconds), but it knows nothing about months or years because those have variable lengths. For "one month later," use `dateutil.relativedelta`:

In [ ]:
from dateutil.relativedelta import relativedelta

start = date(2024, 1, 31)
start + relativedelta(months=1)   # date(2024, 2, 29) — clamps to month-end
start + relativedelta(years=1)    # date(2025, 1, 31)

`timedelta` arithmetic on aware datetimes is correct in absolute terms, but adding `timedelta(days=1)` to a zoned wall-clock time can land on the same instant 23 or 25 hours later if a DST transition occurs. If you want "same wall-clock time tomorrow," do the arithmetic in naive local time and re-localize.

## 6. Time zones in practice

`zoneinfo.ZoneInfo` reads from your system's IANA tz database. On Windows, install the `tzdata` package to get one. The legacy `pytz` library is still common in older code; its `localize()` and `normalize()` calls have no equivalent in `zoneinfo` — `zoneinfo` just works with `replace(tzinfo=...)` and `astimezone(...)`. New code should prefer `zoneinfo`.

Be defensive when parsing local civil times around DST transitions: the "spring forward" hour does not exist, and the "fall back" hour exists twice. Validate or pin the offset (`fold=0`/`fold=1`) at the boundary.

## 7. Pandas: the workhorse for data processing

Pandas wraps datetimes as `Timestamp` (a subclass of `datetime`) and ranges as `DatetimeIndex`. Key operations:

In [ ]:
import pandas as pd

# Convert any column to datetime
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")

# Build a date range
idx = pd.date_range("2024-01-01", "2024-12-31", freq="MS")  # month starts
idx_business = pd.bdate_range("2024-01-01", "2024-12-31")   # business days

# Localize and convert
ts = pd.Timestamp("2024-03-10 09:30")
ts_ny = ts.tz_localize("America/New_York")     # naive -> aware
ts_utc = ts_ny.tz_convert("UTC")               # aware -> aware

# .dt accessor on Series
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["week"]    = df["date"].dt.isocalendar().week

# Resampling (aggregate timestamps into periods)
monthly = df.set_index("date")["value"].resample("MS").sum()

Frequency strings (`"D"`, `"B"`, `"W"`, `"MS"`, `"ME"`, `"QS"`, `"QE"`, `"YS"`, `"YE"`) appear everywhere. `S` suffix means period-start; `E` (or unsuffixed legacy `M`/`Q`/`Y`) means period-end. Recent pandas versions prefer the `S`/`E` suffix.

For accounting-style periods, prefer `pd.Period`:

In [ ]:
p = pd.Period("2024-01", freq="M")     # the month, not an instant
p.start_time, p.end_time               # bounds as Timestamps
df["period"] = df["date"].dt.to_period("M")

`Period` is the pandas object closest in spirit to a TM1 time-dimension element.

## 8. Business calendars and holidays

For business-day arithmetic that respects holidays, `pandas.tseries.offsets.CustomBusinessDay` plus the `holidays` package works well:

In [ ]:
import holidays
from pandas.tseries.offsets import CustomBusinessDay

us = holidays.US(years=[2024, 2025])
bday = CustomBusinessDay(holidays=list(us.keys()))
schedule = pd.date_range("2024-01-01", periods=20, freq=bday)

For fiscal calendars (e.g., a year ending June 30, or 4-4-5 weeks), use `pd.Period(..., freq="Q-JUN")` or build a lookup table; the freq-anchor suffix (`-JUN`) tells pandas where the fiscal year ends. For 4-4-5 retail calendars, a small explicit mapping table is usually clearer than trying to express the rule.

## 9. TM1py: bridging Python time and TM1 time dimensions

TM1 time dimensions store elements as strings — `"2024"`, `"Q1-2024"`, `"Jan-2024"`, `"2024.01"`, `"WK01-2024"` — whose exact format is a modeling choice. Your job is to translate between Python `datetime`/`Period` and these element names.

### 9.1 Generating period element names

A small helper centralizes the convention used in your model:

In [ ]:
import pandas as pd

MONTH_FMT = "%b-%Y"  # e.g., "Jan-2024"; adjust to your TM1 model

def month_element(d) -> str:
    return pd.Timestamp(d).strftime(MONTH_FMT)

def quarter_element(d) -> str:
    ts = pd.Timestamp(d)
    return f"Q{(ts.month - 1) // 3 + 1}-{ts.year}"

def year_element(d) -> str:
    return str(pd.Timestamp(d).year)

Iterate ranges into elements with `pd.date_range`:

In [ ]:
months = [month_element(d) for d in pd.date_range("2024-01-01", "2024-12-01", freq="MS")]
# ['Jan-2024', 'Feb-2024', ..., 'Dec-2024']

Note that `%b` is locale-dependent; if your runtime locale isn't English, build month names from a fixed tuple instead.

### 9.2 Filtering MDX by date range

When pulling a slice from a cube, build the period set in Python and inject it into MDX:

In [ ]:
from TM1py import TM1Service

months = [month_element(d) for d in pd.date_range("2024-01-01", "2024-06-01", freq="MS")]
member_set = ",".join(f"[Period].[{m}]" for m in months)

mdx = f"""
SELECT
  {{ [Measures].[Amount] }} ON COLUMNS,
  NON EMPTY {{ {member_set} }} * {{ [Account].Members }} ON ROWS
FROM [Finance]
"""

with TM1Service(address="localhost", port=8010, user="admin", password="apple", ssl=True) as tm1:
    df = tm1.cubes.cells.execute_mdx_dataframe(mdx)

Building the explicit member set is more reliable than relying on TM1's `:` range operator when element names aren't strictly sortable as strings.

### 9.3 Parsing period elements back to datetimes

Round-tripping the other direction:

In [ ]:
df["period_dt"] = pd.to_datetime(df["Period"], format=MONTH_FMT)
df = df.sort_values("period_dt")

If your model mixes period grains (months, quarters, years all in one dimension), branch on the element pattern or maintain a side table mapping element name to `(start, end, grain)`. Lexical sorting of element strings is almost always wrong — `"Apr-2024"` sorts before `"Jan-2024"`. Always sort by the parsed datetime.

### 9.4 Writing dated values

For point writes, build the element tuple per the dimension order of the cube:

In [ ]:
tm1.cubes.cells.write_value(
    cube_name="Finance",
    value=12345.0,
    element_tuple=("Actual", "USD", "Jan-2024", "4000-Revenue"),
)

For bulk writes from a DataFrame, emit a `{coordinates_tuple: value}` mapping:

In [ ]:
records = {
    ("Actual", "USD", month_element(row["date"]), row["account"]): row["amount"]
    for _, row in df.iterrows()
}
tm1.cubes.cells.write_values("Finance", records)

### 9.5 Audit and transaction logs

The server log methods accept timezone-aware datetimes. TM1 stores log timestamps in UTC; pass aware UTC datetimes and let pandas convert for display:

In [ ]:
from datetime import datetime, timezone, timedelta

since = datetime.now(timezone.utc) - timedelta(days=1)
entries = tm1.server.get_transaction_log_entries(since=since)

log = pd.DataFrame(entries)
log["TimeStamp"] = pd.to_datetime(log["TimeStamp"], utc=True).dt.tz_convert("America/New_York")

### 9.6 A note on cell values that are dates

TM1 cells store numbers and strings. If a model encodes dates inside cells, it's usually one of: an ISO string (`"2024-03-10"`), an Excel serial (`45361`), or a numeric `YYYYMMDD` (`20240310`). Each parses differently:

In [ ]:
pd.to_datetime("2024-03-10")                                  # ISO
pd.to_datetime(45361, unit="D", origin="1899-12-30")          # Excel serial
pd.to_datetime("20240310", format="%Y%m%d")                   # numeric date

Document which convention your cubes use; don't auto-detect.

## 10. Common pitfalls

- **Comparing naive and aware datetimes.** Wrap inputs at the boundary so everything inside your pipeline is one or the other (preferably aware UTC).
- **`%Y` vs `%y`.** Four-digit vs two-digit year. `%y` interprets `24` as `2024` only because of a sliding window; avoid in new code.
- **Locale-dependent codes.** `%b` (`Jan`) and `%B` (`January`) depend on locale. For TM1 element names, set locale explicitly or build month names from a fixed list.
- **End-of-month and DST.** Use `relativedelta` for months/years and aware datetimes around DST transitions.
- **Excel epoch.** Practically `origin="1899-12-30"` in `pd.to_datetime` because of the Lotus 1-2-3 leap-year bug Excel preserved.
- **Sorting period strings.** `"Apr-2024"` sorts before `"Jan-2024"` lexically. Always sort by a parsed datetime, not the element string.
- **Writing back to TM1 with the wrong element name.** A typo in `MONTH_FMT` silently writes to a non-existent (or worse, wrong) element. Validate generated names against `tm1.elements.get_element_names(...)` once at startup.

## 11. A minimal toolkit

For most data-processing work with TM1py, this set of imports covers 95% of cases:

In [ ]:
from datetime import date, datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from dateutil.relativedelta import relativedelta
import pandas as pd
from TM1py import TM1Service

Layer your code so that translation between Python datetimes and TM1 element strings happens in one place — a small helpers module — and the rest of your pipeline can stay in pure pandas/datetime land.